# Clean BCS validation

This notebook compares the finite-bandwidth momentum-space gap equation with independent real-space self-consistent BdG calculations. Units use t=1 and k_B=1; chemical potential is fixed.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from src.lattice import square_lattice_hamiltonian, square_lattice_dispersion, momentum_grid
from src.gap_equation import uniform_gap
from src.solver import solve_bdg
from src.disorder import onsite_disorder
from src.hamiltonian import gaussian_dos
print("Project:", ROOT)


In [ ]:
L, U, T, mu = 4, 2.2, 0.08, -0.5
reference = uniform_gap(L, U, T, mu=mu)
K = square_lattice_hamiltonian(L, mu=mu)
result = solve_bdg(K, U, T, initial_gap=reference, tolerance=1e-9)
print(f"k-space gap={reference:.12g}; BdG mean={result.gap.mean():.12g}")
print("converged:", result.converged, "max spatial variation:", np.ptp(result.gap))
assert result.converged and np.allclose(result.gap, reference, atol=1e-8)


The reference uses the finite periodic lattice k-grid and the same dispersion, chemical potential, and pairing-only assumptions as BdG. It is not a constant-DOS approximation.

In [ ]:
temperatures = np.linspace(0.01, 0.4, 35)
gaps = np.array([uniform_gap(12, U, t, mu=mu) for t in temperatures])
plt.plot(temperatures, gaps, "o-", ms=3); plt.xlabel("T/t"); plt.ylabel("Uniform pairing amplitude Δ/t"); plt.grid(alpha=.3); plt.show()
